In [ ]:
# =============================================================================
# CLASSIFICATION INFERENCE NOTEBOOK
# Input: CSV with SMILES column
# Output: CSV with stability predictions (high/low/medium)
# =============================================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from rdkit import Chem
from rdkit.Chem import Descriptors

print("✓ All imports successful")

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_PATH = '../../../data/003_kaggle_competition_data/25261_regression_sample_submission.csv'
MODEL_PATH = '../../../models/classification_model.joblib'
OUTPUT_PATH = '../../../data/003_kaggle_competition_data/submission_classification.csv'

print(f"Input:  {INPUT_PATH}")
print(f"Model:  {MODEL_PATH}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
print("="*60)
print("LOADING MODEL")
print("="*60)

model_package = joblib.load(MODEL_PATH)

# Extract components
var_sel = model_package['var_sel']
imp = model_package['imp']
sel = model_package['sel']
scaler = model_package['scaler']
le = model_package['le']
model = model_package['model']

print("✓ Model loaded")
print(f"CV F1: {model_package['cv_f1_mean']:.4f} (±{model_package['cv_f1_std']:.4f})")
print(f"Test F1: {model_package['test_f1']:.4f}")
print(f"Classes: {list(le.classes_)}")

In [ ]:
print("="*60)
print("LOADING INPUT DATA")
print("="*60)

df_input = pd.read_csv(INPUT_PATH)

if 'smiles' not in df_input.columns:
    raise ValueError("Input CSV must contain a 'smiles' column")

smiles_list = df_input['smiles'].tolist()

print(f"✓ Loaded {len(smiles_list)} molecules")
print(f"\nFirst 5 SMILES:")
for i, smi in enumerate(smiles_list[:5], 1):
    print(f"  {i}. {smi[:60]}{'...' if len(smi) > 60 else ''}")

In [ ]:
print("="*60)
print("COMPUTING DESCRIPTORS")
print("="*60)

descriptor_names = [x[0] for x in Descriptors._descList]

def compute_descriptors(smiles_str):
    mol = Chem.MolFromSmiles(smiles_str)
    if mol is None:
        return [np.nan] * len(descriptor_names)
    return [getattr(Descriptors, name)(mol) for name in descriptor_names]

X_input = pd.DataFrame(
    [compute_descriptors(s) for s in smiles_list],
    columns=descriptor_names
)

print(f"✓ Descriptors computed: {X_input.shape}")

In [ ]:
print("="*60)
print("PREPROCESSING")
print("="*60)

# Variance threshold
X_clean = var_sel.transform(X_input)
print(f"After variance threshold: {X_clean.shape[1]} features")

# Imputation
X_imp = imp.transform(X_clean)
print(f"After imputation: {X_imp.shape}")

# Feature selection
X_sel = sel.transform(X_imp)
print(f"After feature selection: {X_sel.shape}")

# Scaling
X_scaled = scaler.transform(X_sel)
print(f"After scaling: {X_scaled.shape}")

print("\n✓ Preprocessing complete")

In [ ]:
print("="*60)
print("GENERATING PREDICTIONS")
print("="*60)

# Predict
pred_encoded = model.predict(X_scaled)
pred_labels = le.inverse_transform(pred_encoded)

# Prediction probabilities
pred_proba = model.predict_proba(X_scaled)

# Distribution
pred_dist = dict(zip(*np.unique(pred_labels, return_counts=True)))

print(f"Prediction distribution:")
print(f"  high:   {pred_dist.get('high', 0)} ({pred_dist.get('high', 0)/len(pred_labels)*100:.1f}%)")
print(f"  low:    {pred_dist.get('low', 0)} ({pred_dist.get('low', 0)/len(pred_labels)*100:.1f}%)")
print(f"  medium: {pred_dist.get('medium', 0)} ({pred_dist.get('medium', 0)/len(pred_labels)*100:.1f}%)")

In [ ]:
print("="*60)
print("SAVING SUBMISSION")
print("="*60)

submission = pd.DataFrame({
    'smiles': smiles_list,
    'stability': pred_labels
})

submission.to_csv(OUTPUT_PATH, index=False)

print(f"✓ Saved to: {OUTPUT_PATH}")
print(f"\nFirst 10 predictions:")
print(submission.head(10).to_string(index=False))

In [ ]:
print("="*60)
print("PREDICTION CONFIDENCE ANALYSIS")
print("="*60)

# Get max probability for each prediction
max_proba = pred_proba.max(axis=1)

print(f"Confidence statistics:")
print(f"  Mean: {max_proba.mean():.4f}")
print(f"  Std:  {max_proba.std():.4f}")
print(f"  Min:  {max_proba.min():.4f}")
print(f"  Max:  {max_proba.max():.4f}")

# Confidence buckets
high_conf = (max_proba >= 0.8).sum()
med_conf = ((max_proba >= 0.5) & (max_proba < 0.8)).sum()
low_conf = (max_proba < 0.5).sum()

print(f"\nConfidence distribution:")
print(f"  High (≥0.8):   {high_conf} ({high_conf/len(max_proba)*100:.1f}%)")
print(f"  Medium (0.5-0.8): {med_conf} ({med_conf/len(max_proba)*100:.1f}%)")
print(f"  Low (<0.5):    {low_conf} ({low_conf/len(max_proba)*100:.1f}%)")